# ModSSC | Official vs custom splits

Compare the default sampling behavior that respects provider test splits with the override mode that applies a user split to `dataset.train`.

## Objective
- Load a dataset with an official test split.
- Sample once with `respect_official_test=True`.
- Sample again with `allow_override_official=True`.


In [ ]:
from modssc.data_loader import load_dataset
from modssc.sampling import HoldoutSplitSpec, LabelingSpec, SamplingPlan, SamplingPolicy, sample

ds = load_dataset("toy", download=True)
dataset_fingerprint = str(ds.meta["dataset_fingerprint"])

base_split = HoldoutSplitSpec(test_fraction=0.0, val_fraction=0.2, stratify=True)
labeling = LabelingSpec(mode="fraction", value=0.2, strategy="balanced")

In [ ]:
plan_official = SamplingPlan(
    split=base_split,
    labeling=labeling,
    policy=SamplingPolicy(respect_official_test=True, allow_override_official=False),
)
plan_override = SamplingPlan(
    split=base_split,
    labeling=labeling,
    policy=SamplingPolicy(respect_official_test=True, allow_override_official=True),
)

res_official, _ = sample(
    ds, plan=plan_official, seed=0, dataset_fingerprint=dataset_fingerprint, save=False
)
res_override, _ = sample(
    ds, plan=plan_override, seed=0, dataset_fingerprint=dataset_fingerprint, save=False
)

{
    "official": {
        "refs": dict(res_official.refs),
        "policy": res_official.stats["policy"],
        "sizes": {
            k: int(v.size) for k, v in res_official.indices.items() if k in {"train", "val", "test"}
        },
    },
    "override": {
        "refs": dict(res_override.refs),
        "policy": res_override.stats["policy"],
        "sizes": {
            k: int(v.size) for k, v in res_override.indices.items() if k in {"train", "val", "test"}
        },
    },
}